## ra_dec_offset_v6
### AdHoc Query notebook
# version notes
- v5 updated for adhoc queries to add to nina plugin csv
- v6 Removed the index column by setting it as "Name*"
- v6 Set the value Temp to '' this is a column for Template

In [1]:
#!pip install wikipedia

In [2]:
#!pip install --upgrade astroquery

In [3]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki

In [4]:
# ra_dec_offset_v5 and output files naming
ra_dec_offset_version = "v6"
data_folder_name = "data_folder"
output_csvfilename = f"adhoc_ra_dec_offset_{ra_dec_offset_version}.csv"
print(f"ra_dec_offset_version is: {ra_dec_offset_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

ra_dec_offset_version is: v6
data_folder_name is: data_folder
output_csvfilename is: adhoc_ra_dec_offset_v6.csv


In [5]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")


Observers Location is: San Diego
Observers Telescope is: BARO
Observers Lattitude is: 32.6 deg
Observers Longitude is: -116.3 deg
Observers Height is: 1131.0 m
Safe Limit for BARO is: 10.0 deg
Max Mag to Query is: 8
Min RA to Query is: 12
Max RA to Query is: 18
Altitude: 7.1947 deg
Azimuth: 289.3402 deg
Observable Declination range: -47d24m00s to 112d36m00s


In [6]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

The Zenith at San Diego is: 57.40 deg deg
Safe Declination limits at BARO are: -47.40 deg deg to 112.60 deg deg


In [7]:
# set target default name
target_default_name = "HD"

In [8]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [9]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    return new_coord_



In [10]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

In [11]:
print(df)

Empty DataFrame
Columns: [Name1*, Name2*, RA2000*, D2000*, Pmag~, Exp~, Note1, Note2, NExp~, GetRef, Temp]
Index: []


In [12]:
# --- Change User Inputs ---
target_name = "Denebola"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (177.26490976, 14.57205807)>
Using SkyOffsetFrame for Star Denebola 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.817661, 14.572058
Original RA(Deg)/Dec: 177.264910, 14.572058
New  RA(Hr)/Dec:  11.821388, 14.550294
New  RA(Deg)/Dec:  177.320827, 14.550294
Delta RA/DEC(min): -3.355031,       1.305856


In [13]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Denebola
https://en.wikipedia.org/wiki/Denebola#Nomenclature
https://en.wikipedia.org/wiki/Denebola#Properties
https://en.wikipedia.org/wiki/Denebola#References
https://en.wikipedia.org/wiki/Denebola_(disambiguation)
https://simple.wikipedia.org/wiki/Leo_(constellation)
https://en.wikipedia.org/wiki/Deneb
https://en.wikipedia.org/wiki/Leo_(constellation)
https://the-universe-of-the-universe.fandom.com/wiki/Denebola
https://www.facebook.com/photo.php?fbid=1025645050877279&id=406815242760266&set=a.832754770166309


In [14]:
star_magnitude = 2.14;  target_alt_name = "HD 102647" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.14: 3.98 s


In [15]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*     RA2000*     D2000* Pmag~  Exp~  \
0  Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294  2.14  3.98   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       


In [16]:
# --- Change User Inputs ---
target_name = "Arcturus"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (213.9153003, 19.18240916)>
Using SkyOffsetFrame for Star Arcturus 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.261020, 19.182409
Original RA(Deg)/Dec: 213.915300, 19.182409
New  RA(Hr)/Dec:  14.264840, 19.160643
New  RA(Deg)/Dec:  213.972598, 19.160643
Delta RA/DEC(min): -3.437878,       1.305991


In [17]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://upload.wikimedia.org/wikipedia/commons/4/47/Bo%C3%B6tes_IAU.svg?sa=X&ved=2ahUKEwiYtNv6-4CPAxWuEzQIHa7PAJ0Q_B16BAgJEAI
https://en.wikipedia.org/wiki/Arcturus
https://simple.wikipedia.org/wiki/Arcturus
https://en.wikipedia.org/wiki/Arcturus_moving_group
https://commons.wikimedia.org/wiki/File:Arcturus_(optical).png
https://the-universe-of-the-universe.fandom.com/wiki/Arcturus
https://en.wikipedia.org/wiki/Spring_Triangle
https://www.space.com/22842-arcturus.html
https://chaos-chronicles.fandom.com/wiki/Arcturus
https://en.wikipedia.org/wiki/Arcturus_(disambiguation)


In [18]:
star_magnitude = -0.05;  target_alt_name = 'HD 124897' # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag -0.05: 0.58 s


In [19]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*     RA2000*     D2000*  Pmag~  Exp~  \
0  Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14  3.98   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05  0.58   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       


In [20]:
# --- Change User Inputs ---
target_name = "Neptune"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (0.12937671, -0.63449956)>
Using SkyOffsetFrame for Star Neptune 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.008625, -0.634500
Original RA(Deg)/Dec: 0.129377, -0.634500
New  RA(Hr)/Dec:  0.012234, -0.656257
New  RA(Deg)/Dec:  0.183504, -0.656257
Delta RA/DEC(min): -3.247640,       1.305440


In [21]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Outline_of_Neptune
https://en.wikipedia.org/wiki/Neptune
https://en.wikipedia.org/wiki/Hot_Neptune
https://en.wikipedia.org/wiki/Neptune_(mythology)
https://en.wikipedia.org/wiki/Neptune_(disambiguation)
https://en.wikipedia.org/wiki/Discovery_of_Neptune
https://en.wikipedia.org/wiki/Moons_of_Neptune
https://en.wikipedia.org/wiki/Proteus_(moon)
https://en.wikipedia.org/wiki/Nereid_(moon)
https://en.wikipedia.org/wiki/S/2021_N_1


In [22]:
star_magnitude = 7.74;  target_alt_name = target_default_name # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'NA'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 7.74: 556.18 s


In [23]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*     RA2000*     D2000*  Pmag~    Exp~  \
0  Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14    3.98   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05    0.58   
2          Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74  556.18   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       


In [24]:
# --- Change User Inputs ---
target_name = "Zosma"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (168.52708927, 20.52371814)>
Using SkyOffsetFrame for Star Zosma 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.235139, 20.523718
Original RA(Deg)/Dec: 168.527089, 20.523718
New  RA(Hr)/Dec:  11.238992, 20.501951
New  RA(Deg)/Dec:  168.584873, 20.501951
Delta RA/DEC(min): -3.467027,       1.306031


In [25]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Delta_Leonis
https://en.wikipedia.org/wiki/Leo_(constellation)
https://the-universe-of-the-universe.fandom.com/wiki/Zosma
https://wiki.ed-board.net/en/object/zosma
https://en.wiktionary.org/wiki/Zosma
https://wikisky.org/starview?object_type=1&object_id=296&object_name=HIP+54872&locale=CA
https://en.wikipedia.org/wiki/NGC_3632
http://stars.astro.illinois.edu/sow/zosma.html
https://aquarii.fandom.com/wiki/Zosma
https://library.keplercollege.org/fall-madona/


In [26]:
star_magnitude = 2.56;  target_alt_name = "HD 97603" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A4'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.56: 5.76 s


In [27]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                      Name1*     Name2*     RA2000*     D2000*  Pmag~    Exp~  \
0  Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14    3.98   
1  Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05    0.58   
2          Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74  556.18   
3      Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56    5.76   

  Note1 Note2 NExp~ GetRef Temp  
0    NA    NA     1      0       
1    NA    NA     1      0       
2    NA    NA     1      0       
3    NA    NA     1      0       


In [28]:
star_magnitude = 2.56;  target_alt_name = target_default_name # FIX THIS LINE AND BELOW BASED ON SEARCH
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.56: 5.76 s


In [29]:
# --- Change User Inputs ---
target_name = "Minelauva"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (193.90086927, 3.3974689)>
Using SkyOffsetFrame for Star Minelauva 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.926725, 3.397469
Original RA(Deg)/Dec: 193.900869, 3.397469
New  RA(Hr)/Dec:  12.930339, 3.375710
New  RA(Deg)/Dec:  193.955087, 3.375710
Delta RA/DEC(min): -3.253071,       1.305548


In [30]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Delta_Virginis
https://en.wikipedia.org/wiki/Delta_Virginis#Nomenclature
https://en.wikipedia.org/wiki/Delta_Virginis#Properties
https://en.wikipedia.org/wiki/Delta_Virginis#Substellar_companion
https://the-universe-of-the-universe.fandom.com/wiki/Minelauva_A
https://next-generation-astronomy.fandom.com/wiki/Minelauva
https://en.wikipedia.org/wiki/List_of_proper_names_of_stars
https://en.wikipedia.org/wiki/Hebrew_astronomy
https://en.wikipedia.org/wiki/Vela_(constellation)
https://en.wikipedia.org/wiki/Lynx_(constellation)


In [31]:
star_magnitude = 3.4;  target_alt_name = "HD 112300" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.4: 12.08 s


In [32]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA    NA     1      0       
4   12.08    NA    NA     1      0       


In [33]:
# --- Change User Inputs ---
target_name = "HD 138629"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (232.94576269, 40.89933486)>
Using SkyOffsetFrame for Star HD 138629 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.529718, 40.899335
Original RA(Deg)/Dec: 232.945763, 40.899335
New  RA(Hr)/Dec:  15.534490, 40.877555
New  RA(Deg)/Dec:  233.017345, 40.877555
Delta RA/DEC(min): -4.294914,       1.306785


In [34]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Nu2_Bo%C3%B6tis
https://www.wikisky.org/group?id=23&page=members&locale=ZH
https://en.wikipedia.org/wiki/Henry_Draper_Catalogue
https://server6.wikisky.org/starview?object_type=1&object_id=1861&object_name=%CE%BD%C2%B2+Boo&locale=TR
https://ast.wikipedia.org/wiki/Ni2_Bootis
https://en.namu.wiki/w/%EB%AA%A9%EB%8F%99%EC%9E%90%EB%A6%AC%20%EB%88%84%C2%B2
https://es.wikipedia.org/wiki/Ni2_Bootis
https://ru.wikipedia.org/wiki/%D0%9D%D1%8E%C2%B2_%D0%92%D0%BE%D0%BB%D0%BE%D0%BF%D0%B0%D1%81%D0%B0
https://en.wikipedia.org/wiki/Astronomy
https://en.wikipedia.org/wiki/History_of_astronomy


In [35]:
star_magnitude = 5.02;  target_alt_name = "Nu2 Boo" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A5'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 5.02: 50.46 s


In [36]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5  HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA    NA     1      0       
4   12.08    NA    NA     1      0       
5   50.46    NA    NA     1      0       


In [37]:
# --- Change User Inputs ---
target_name = "HD 142105"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (236.01466071, 77.79449312)>
Using SkyOffsetFrame for Star HD 142105 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.734311, 77.794493
Original RA(Deg)/Dec: 236.014661, 77.794493
New  RA(Hr)/Dec:  15.751348, 77.772618
New  RA(Deg)/Dec:  236.270213, 77.772618
Delta RA/DEC(min): -15.333150,       1.312536


In [38]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Zeta_Ursae_Minoris
https://en.wikipedia.org/wiki/HD_142
https://server5.wikisky.org/group?locale=TR&id=23
https://www.wikisky.org/starview?object_type=1&object_id=929&object_name=16+UMi&locale=ES
https://en.wikipedia.org/wiki/Henry_Draper_Catalogue
https://en.wikipedia.org/wiki/HD_140283
https://en.wikipedia.org/wiki/List_of_oldest_stars
https://en.wikipedia.org/wiki/Sneden%27s_Star
https://en.wikipedia.org/wiki/Subgiant
https://en.wikipedia.org/wiki/Galactic_halo


In [39]:
star_magnitude = 4.29;  target_alt_name = "Zeta Umi" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.29: 26.50 s


In [40]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5  HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6  HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA    NA     1      0       
4   12.08    NA    NA     1      0       
5   50.46    NA    NA     1      0       
6   26.50    NA    NA     1      0       


In [41]:
# --- Change User Inputs ---
target_name = "R Lyr"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (283.83375974, 43.94608958)>
Using SkyOffsetFrame for Star R Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.922251, 43.946090
Original RA(Deg)/Dec: 283.833760, 43.946090
New  RA(Hr)/Dec:  18.927260, 43.924307
New  RA(Deg)/Dec:  283.908905, 43.924307
Delta RA/DEC(min): -4.508707,       1.306935


In [42]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/R_Lyrae
https://en.wikipedia.org/wiki/RR_Lyrae
https://en.wikipedia.org/wiki/Lyra
https://en.wikipedia.org/wiki/List_of_stars_in_Lyra
https://en.wikipedia.org/wiki/Lyra_(disambiguation)
https://en.wikipedia.org/wiki/Delta2_Lyrae
https://en.wikipedia.org/wiki/RR_Lyrae_variable
https://wikipedia.nucleos.com/viewer/wikipedia_en_all/A/R_Lyrae
https://en.wikipedia.org/wiki/List_of_variable_stars
https://en.wikipedia.org/wiki/Lyrids


In [43]:
star_magnitude = 3.9;  target_alt_name = "HD 175865" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.9: 18.79 s


In [44]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5  HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6  HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA    NA     1      0       
4   12.08    NA    NA     1      0       
5   50.46    NA    NA     1      0       
6   26.50    NA    NA     1      0     

In [45]:
# --- Change User Inputs ---
target_name = "Zet1 Lyr"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (281.19315451, 37.60512165)>
Using SkyOffsetFrame for Star Zet1 Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.746210, 37.605122
Original RA(Deg)/Dec: 281.193155, 37.605122
New  RA(Hr)/Dec:  18.750763, 37.583344
New  RA(Deg)/Dec:  281.261452, 37.583344
Delta RA/DEC(min): -4.097870,       1.306638


In [46]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Zeta1_Lyrae
https://en.wikipedia.org/wiki/Lyra
https://en.wikipedia.org/wiki/List_of_stars_in_Lyra
https://en.wikipedia.org/wiki/Lyra_(disambiguation)
https://en.wikipedia.org/wiki/R_Lyrae
https://en.wikipedia.org/wiki/Delta2_Lyrae
https://fr.wikipedia.org/wiki/Zeta1_Lyrae
https://en.wikipedia.org/wiki/Zeta2_Lyrae
https://en.wikipedia.org/wiki/List_of_nearest_stars
https://en.wikipedia.org/wiki/Barnard%27s_Star


In [47]:
star_magnitude = 4.37;  target_alt_name = "HD 173648" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'NA'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.37: 28.44 s


In [48]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5  HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6  HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8   Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA    NA     1      0       
4   12.08    NA    NA     1      0       
5   50.

In [49]:
# --- Change User Inputs ---
target_name = "P Cyg"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (304.44667489, 38.03293031)>
Using SkyOffsetFrame for Star P Cyg 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.296445, 38.032930
Original RA(Deg)/Dec: 304.446675, 38.032930
New  RA(Hr)/Dec:  20.301025, 38.011153
New  RA(Deg)/Dec:  304.515369, 38.011153
Delta RA/DEC(min): -4.121671,       1.306657


In [50]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/P_Cygni
https://en.wikipedia.org/wiki/P_Cygni#Visibility
https://en.wikipedia.org/wiki/P_Cygni#Luminous_blue_variable
https://en.wikipedia.org/wiki/P_Cygni#Evolution
https://en.wikipedia.org/wiki/P_Cygni#P_Cygni_profile
https://en.wikipedia.org/wiki/Pi1_Cygni
https://en.wikipedia.org/wiki/Cygnus_(constellation)
https://en.wikipedia.org/wiki/Pi2_Cygni
https://en.wikipedia.org/wiki/Pi_Cygni
https://en.wikipedia.org/wiki/Wolf%E2%80%93Rayet_star


In [51]:
star_magnitude = 4.82;  target_alt_name = "HD 193237" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 4.82: 42.30 s


In [52]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                       Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0   Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1   Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2           Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3       Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4  Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5  HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6  HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7      R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8   Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9      P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   

     Exp~ Note1 Note2 NExp~ GetRef Temp  
0    3.98    NA    NA     1      0       
1    0.58    NA    NA     1      0       
2  556.18    NA    NA     1      0       
3    5.76    NA  

In [53]:
# --- Change User Inputs ---
target_name = "Navi"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (22.18838359, -1.87361633)>
Using SkyOffsetFrame for Star Navi 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 1.479226, -1.873616
Original RA(Deg)/Dec: 22.188384, -1.873616
New  RA(Hr)/Dec:  1.482836, -1.895373
New  RA(Deg)/Dec:  22.242537, -1.895373
Delta RA/DEC(min): -3.249204,       1.305407


In [54]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Gamma_Cassiopeiae
https://en.wikipedia.org/wiki/Gamma_Cassiopeiae#Physical_properties
https://en.wikipedia.org/wiki/Gamma_Cassiopeiae#X-ray_emission
https://en.wikipedia.org/wiki/Gamma_Cassiopeiae#Companions
https://en.wikipedia.org/wiki/Gamma_Cassiopeiae#Names
https://en.wikipedia.org/wiki/Celestial_navigation
https://en.wikipedia.org/wiki/Celestial_navigation#Example
https://en.wikipedia.org/wiki/Celestial_navigation#Angular_measurement
https://en.wikipedia.org/wiki/Celestial_navigation#Practical_navigation
https://en.wikipedia.org/wiki/Celestial_navigation#Modern_celestial_navigation


In [55]:
star_magnitude = 2.47;  target_alt_name = "HD 5394" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B0'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.47: 5.32 s


In [56]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0    Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2            Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5   HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6   HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   
10         Navi_HD 5394_Typ_B0    HD 5394   22.242537  -1.895373   2.47   

      Exp~ Note1 Note2 NExp~ GetRef Temp  
0     3.98    NA    NA     1      0       
1     0.58   

In [57]:
# --- Change User Inputs ---
target_name = "Albireo"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.512021, 27.959674
Original RA(Deg)/Dec: 292.680315, 27.959674
New  RA(Hr)/Dec:  19.516105, 27.937902
New  RA(Deg)/Dec:  292.741579, 27.937902
Delta RA/DEC(min): -3.675822,       1.306271


In [58]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://upload.wikimedia.org/wikipedia/commons/thumb/f/f5/NewAlbireo.jpg/250px-NewAlbireo.jpg?sa=X&ved=2ahUKEwi2-KGH_ICPAxVjOzQIHRq0Je4Q_B16BAgJEAI
https://en.wikipedia.org/wiki/Albireo
https://simple.wikipedia.org/wiki/Albireo
https://cosmos-universe.fandom.com/wiki/Albireo
https://en.wikipedia.org/wiki/Asterism_(astronomy)
https://earthsky.org/brightest-stars/albireo-finest-double-star/
https://en.wikipedia.org/wiki/Double_star
http://stars.astro.illinois.edu/sow/albireo.html
https://www.syfy.com/syfy-wire/long-standing-astronomical-mystery-solved-albireo-is-not-a-binary-star
http://www.waloszek.de/inhalt_astro_dso_albireo_e.html


In [59]:
star_magnitude = 3.21;  target_alt_name = "HD 183912" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 3.21: 10.22 s


In [60]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0    Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2            Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5   HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6   HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   
10         Navi_HD 5394_Typ_B0    HD 5394   22.242537  -1.895373   2.47   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.741579  27.937902   3.21   

      Exp~ Note1 Note2 N

In [61]:
star_magnitude = 5.11;  target_alt_name = "HD 183913" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 5.11: 54.63 s


In [62]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0    Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2            Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5   HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6   HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   
10         Navi_HD 5394_Typ_B0    HD 5394   22.242537  -1.895373   2.47   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.741579  27.937902   3.21   
12    Albireo_HD 183913_T

In [63]:
# --- Change User Inputs ---
target_name = "Alioth"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (193.50728997, 55.95982296)>
Using SkyOffsetFrame for Star Alioth 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.900486, 55.959823
Original RA(Deg)/Dec: 193.507290, 55.959823
New  RA(Hr)/Dec:  12.906928, 55.938028
New  RA(Deg)/Dec:  193.603924, 55.938028
Delta RA/DEC(min): -5.798054,       1.307727


In [64]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Alioth
https://en.wikipedia.org/wiki/Alioth#Physical_characteristics
https://en.wikipedia.org/wiki/Alioth#Name_and_etymology
https://simple.wikipedia.org/wiki/Alioth
https://simple.wikipedia.org/wiki/Epsilon_Ursae_Majoris
https://wiki.alioth.net/index.php/Planetary_Systems
https://en.wikipedia.org/wiki/Ursa_Major
https://en.wikipedia.org/wiki/List_of_stars_in_Ursa_Major
https://en.wikipedia.org/wiki/Ursa_Major_(disambiguation)
https://en.wikipedia.org/wiki/Ursa_Minor


In [65]:
star_magnitude = 1.77;  target_alt_name = "HD 112185" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A1'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 1.77: 2.87 s


In [66]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0    Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2            Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5   HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6   HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   
10         Navi_HD 5394_Typ_B0    HD 5394   22.242537  -1.895373   2.47   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.741579  27.937902   3.21   
12    Albireo_HD 183913_T

In [67]:
# --- Change User Inputs ---
target_name = "Vega"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (279.23473479, 38.78368896)>
Using SkyOffsetFrame for Star Vega 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.615649, 38.783689
Original RA(Deg)/Dec: 279.234735, 38.783689
New  RA(Hr)/Dec:  18.620276, 38.761911
New  RA(Deg)/Dec:  279.304146, 38.761911
Delta RA/DEC(min): -4.164680,       1.306689


In [68]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Vega
https://en.wikipedia.org/wiki/Vega#Observational_history
https://en.wikipedia.org/wiki/Vega#Physical_characteristics
https://en.wikipedia.org/wiki/Vega#Possible_planetary_system
https://simple.wikipedia.org/wiki/Vega
https://en.wikiversity.org/wiki/Stars/Vega
https://en.wikipedia.org/wiki/Category:Vega
https://astronomical.fandom.com/wiki/Vega
https://en.wikipedia.org/wiki/Summer_Triangle
https://en.wikipedia.org/wiki/Stellar_classification


In [69]:
star_magnitude = 0.02;  target_alt_name = "HD 172167" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 0.02: 0.61 s


In [70]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1
print(df)

                        Name1*     Name2*     RA2000*     D2000*  Pmag~  \
0    Denebola_HD 102647_Typ_A3  HD 102647  177.320827  14.550294   2.14   
1    Arcturus_HD 124897_Typ_K1  HD 124897  213.972598  19.160643  -0.05   
2            Neptune_HD_Typ_NA         HD    0.183504  -0.656257   7.74   
3        Zosma_HD 97603_Typ_A4   HD 97603  168.584873  20.501951   2.56   
4   Minelauva_HD 112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   
5   HD 138629_HD 138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   
6   HD 142105_HD 142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   
7       R Lyr_HD 175865_Typ_M5  HD 175865  283.908905  43.924307    3.9   
8    Zet1 Lyr_HD 173648_Typ_NA  HD 173648  281.261452  37.583344   4.37   
9       P Cyg_HD 193237_Typ_B1  HD 193237  304.515369  38.011153   4.82   
10         Navi_HD 5394_Typ_B0    HD 5394   22.242537  -1.895373   2.47   
11    Albireo_HD 183912_Typ_K2  HD 183912  292.741579  27.937902   3.21   
12    Albireo_HD 183913_T

In [71]:
# --- Change User Inputs ---
target_name = "Scheat"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (345.94357274, 28.08278712)>
Using SkyOffsetFrame for Star Scheat 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 23.062905, 28.082787
Original RA(Deg)/Dec: 345.943573, 28.082787
New  RA(Hr)/Dec:  23.066994, 28.061016
New  RA(Deg)/Dec:  346.004906, 28.061016
Delta RA/DEC(min): -3.680024,       1.306275


In [72]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/Beta_Pegasi
https://en.wikipedia.org/wiki/Pegasus_(constellation)
https://the-universe-of-the-universe.fandom.com/wiki/Scheat
https://en.wikipedia.org/wiki/List_of_largest_stars
https://en.wikipedia.org/wiki/RSGC1
https://en.wikipedia.org/wiki/VY_Canis_Majoris
https://en.wikipedia.org/wiki/Stephenson_2
https://en.wikipedia.org/wiki/R_Doradus
https://en.wikipedia.org/wiki/List_of_Arabic_star_names
https://en.wiktionary.org/wiki/Scheat


In [73]:
star_magnitude = 2.42;  target_alt_name = "HD 217906" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 2.42: 5.09 s


In [74]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1

In [75]:
df['Name1*'] = df['Name1*'].str.replace(' ', '_')
mdf = df.set_index("Name1*")
mdf.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(mdf)

                               Name2*     RA2000*     D2000*  Pmag~    Exp~  \
Name1*                                                                        
Denebola_HD_102647_Typ_A3   HD 102647  177.320827  14.550294   2.14    3.98   
Arcturus_HD_124897_Typ_K1   HD 124897  213.972598  19.160643  -0.05    0.58   
Neptune_HD_Typ_NA                  HD    0.183504  -0.656257   7.74  556.18   
Zosma_HD_97603_Typ_A4        HD 97603  168.584873  20.501951   2.56    5.76   
Minelauva_HD_112300_Typ_M3  HD 112300  193.955087   3.375710    3.4   12.08   
HD_138629_HD_138629_Typ_A5  HD 138629  233.017345  40.877555   5.02   50.46   
HD_142105_HD_142105_Typ_A3  HD 142105  236.270213  77.772618   4.29   26.50   
R_Lyr_HD_175865_Typ_M5      HD 175865  283.908905  43.924307    3.9   18.79   
Zet1_Lyr_HD_173648_Typ_NA   HD 173648  281.261452  37.583344   4.37   28.44   
P_Cyg_HD_193237_Typ_B1      HD 193237  304.515369  38.011153   4.82   42.30   
Navi_HD_5394_Typ_B0           HD 5394   22.242537  -

In [76]:
# --- Change User Inputs ---
target_name = "RR Lyra"       # Your target star
offset_arcmin = -3.5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1: Look up target coordinates ---
original_coord = SkyCoord.from_name(target_name)

print(original_coord)

# --- Step 1A-5 : Call function to calculate adjusted coordinates ---
new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)

# --- Step 6: Report result ---
print(f"Using SkyOffsetFrame for Star {target_name} ")
print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.6f}, {original_coord.dec.deg:.6f}")
print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.6f}, {original_coord.dec.deg:.6f}")
print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.6f}, {new_coord.dec.deg:.6f}")
print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.6f}, {new_coord.dec.deg:.6f}")
print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg)*60:.6f}, \
      {(original_coord.dec.deg-new_coord.dec.deg)*60:.6f}")



<SkyCoord (ICRS): (ra, dec) in deg
    (291.366304, 42.78435924)>
Using SkyOffsetFrame for Star RR Lyra 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.424420, 42.784359
Original RA(Deg)/Dec: 291.366304, 42.784359
New  RA(Hr)/Dec:  19.429335, 42.762578
New  RA(Deg)/Dec:  291.440025, 42.762578
Delta RA/DEC(min): -4.423242,       1.306876


In [77]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_name} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)

https://en.wikipedia.org/wiki/RR_Lyrae
https://en.wikipedia.org/wiki/RR_Lyrae_variable
https://en.wikipedia.org/wiki/RR_Lyrae_variable#Discovery_and_recognition
https://en.wikipedia.org/wiki/RR_Lyrae_variable#Distribution
https://en.wikipedia.org/wiki/RR_Lyrae_variable#Properties
https://en.wikipedia.org/wiki/RR_Lyrae_variable#Period-luminosity_relationships
https://simple.wikipedia.org/wiki/RR_Lyrae
https://en.wikipedia.org/wiki/Lyra
https://en.wikipedia.org/wiki/List_of_stars_in_Lyra
https://en.wikipedia.org/wiki/Lyra_(disambiguation)


In [78]:
star_magnitude = 7.5;  target_alt_name = "HD 182989" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7_F8'
exposure = compute_exposure_time(star_magnitude)
print(f"Example: Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Example: Required exposure for mag 7.5: 450.04 s


In [79]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.6f}',f'{new_coord.dec.deg:.6f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','']
ridx += 1

In [80]:
df['Name1*'] = df['Name1*'].str.replace(' ', '_')
mdf = df.set_index("Name1*")
mdf.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(mdf)

                                Name2*     RA2000*     D2000*  Pmag~    Exp~  \
Name1*                                                                         
Denebola_HD_102647_Typ_A3    HD 102647  177.320827  14.550294   2.14    3.98   
Arcturus_HD_124897_Typ_K1    HD 124897  213.972598  19.160643  -0.05    0.58   
Neptune_HD_Typ_NA                   HD    0.183504  -0.656257   7.74  556.18   
Zosma_HD_97603_Typ_A4         HD 97603  168.584873  20.501951   2.56    5.76   
Minelauva_HD_112300_Typ_M3   HD 112300  193.955087   3.375710    3.4   12.08   
HD_138629_HD_138629_Typ_A5   HD 138629  233.017345  40.877555   5.02   50.46   
HD_142105_HD_142105_Typ_A3   HD 142105  236.270213  77.772618   4.29   26.50   
R_Lyr_HD_175865_Typ_M5       HD 175865  283.908905  43.924307    3.9   18.79   
Zet1_Lyr_HD_173648_Typ_NA    HD 173648  281.261452  37.583344   4.37   28.44   
P_Cyg_HD_193237_Typ_B1       HD 193237  304.515369  38.011153   4.82   42.30   
Navi_HD_5394_Typ_B0            HD 5394  